In [1]:
import ants
import SimpleITK as sitk
import nibabel as nib
import numpy as np
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

import os
import time

os.environ['NEURITE_BACKEND'] = 'pytorch'
os.environ['VXM_BACKEND'] = 'pytorch'
import voxelmorph as vxm
import torch

import shutil

In [ ]:
""" Instance-specific Optimization
generating optimized_atlas_*.pt for inference 
"""
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# prepare model folder
model_dir = 'mice_dataset2/test_4_optimization'
os.makedirs(model_dir, exist_ok=True)

# device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# enabling cudnn determinism appears to speed up training by a lot
torch.backends.cudnn.deterministic = True

# training parameters
int_steps = 7
int_downsize = 2
lr = 1e-4
weight = 1
initial_epoch = 0
epochs = 300
steps_per_epoch = 1

# unet architecture
enc_nf = [16, 32, 32, 32]
dec_nf = [32, 32, 32, 32, 32, 16, 16]


model = vxm.networks.VxmDense(
    inshape=(160, 208, 112),  # input shape
    nb_unet_features=[enc_nf, dec_nf],
    bidir=False,
    int_steps=int_steps,
    int_downsize=int_downsize,
    )
model.to(device)

moving = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/323/normalized_atlas_323.nii', add_batch_axis=True, add_feat_axis=True)
fixed, fixed_affine = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/323/normalized_pa_323.nii', add_batch_axis=True, add_feat_axis=True, ret_affine=True)
input_moving = torch.from_numpy(moving).float().permute(0, 4, 1, 2, 3)
input_fixed = torch.from_numpy(fixed).float().permute(0, 4, 1, 2, 3)
inputs = [input_moving, input_fixed]
outputs = [input_fixed]
outputs.append(torch.from_numpy(np.zeros((1, 160, 208, 112, 1))).float().permute(0, 4, 1, 2, 3))

model_inputs = [inputs, outputs]
model_input, model_y_true = model_inputs

# prepare the model for training and send to device
#model = vxm.networks.VxmDense.load('mice_dataset2/vxm_evaluation_new/1e4_batch4/1000.pt', device)
#model.to(device)
# set optimizer

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# prepare image loss
image_loss_func = vxm.losses.NCC().loss
losses = [image_loss_func]
weights = [1]

# prepare deformation loss
losses += [vxm.losses.Grad('l2', loss_mult=int_downsize).loss]
weights += [weight]

#train
epoch_total_loss_list = []
epoch_similarity_loss_list = []
epoch_deformation_loss_list = []
# training loops

for epoch in range(initial_epoch, epochs):

    model.train()

    epoch_loss = []
    epoch_total_loss = []
    epoch_step_time = []
    

    for step in range(steps_per_epoch):

        step_start_time = time.time()

        # generate inputs (and true outputs) and convert them to tensors

        inputs = [item.to(device) for item in model_input]
        y_true = [item.to(device) for item in model_y_true]
        data_loading_time = time.time()
        print(f"load data to device took{data_loading_time-step_start_time: .2f} seconds")

        # run inputs through the model to produce a warped image and flow field
        y_pred = model(*inputs)
        go_through_model_time = time.time()
        print(f"go through the model took {go_through_model_time-data_loading_time: .2f} seconds")

        # calculate total loss
        loss = 0
        loss_list = []
        for n, loss_function in enumerate(losses):
            curr_loss = loss_function(y_true[n], y_pred[n]) * weights[n]
            loss_list.append(curr_loss.item())
            loss += curr_loss
        epoch_similarity_loss_list.append(loss_list[0])
        epoch_deformation_loss_list.append(loss_list[1])
        loss_calculation_time = time.time()
        print(f"loss calculationt took {loss_calculation_time-go_through_model_time: .2f} seconds")

        epoch_loss.append(loss_list)
        epoch_total_loss.append(loss.item())
        epoch_total_loss_list.append(loss.item())

        # backpropagate and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        backpropagate_time = time.time()
        print(f"backpropagate took {backpropagate_time-loss_calculation_time: .2f} seconds")


        # get compute time
        epoch_step_time.append(time.time() - step_start_time)
    

    # print epoch info
    epoch_info = 'Epoch %d/%d' % (epoch + 1, epochs)
    time_info = 'training %.4f sec/step' % np.mean(epoch_step_time)
    losses_info = ', '.join(['%.4e' % f for f in np.mean(epoch_loss, axis=0)])
    loss_info = 'loss: %.4e  (%s)' % (np.mean(epoch_total_loss), losses_info)
    print(' - '.join((epoch_info, time_info, loss_info)), flush=True)
# final model save
model.save('mice_dataset2/vxm_evaluation_new/323/model_instance_specific_optimized_atlas_323.pt')

In [11]:
""" Generating predicted atlas/warpfield
predicted_atlas: predicted_atals_*.nii
predicted_atlas_warpfield: predicted_atlas_warpfield_*.nii
"""

# device handling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#register
#load moving and fixed images
moving = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/323/normalized_atlas_323.nii', add_batch_axis=True, add_feat_axis=True)
fixed, fixed_affine = vxm.py.utils.load_volfile('mice_dataset2/vxm_evaluation_new/323/normalized_pa_323.nii', add_batch_axis=True, add_feat_axis=True, ret_affine=True)

# load and set up model
model_evaluate = vxm.networks.VxmDense.load('mice_dataset2/vxm_evaluation_new/323/model_instance_specific_optimized_atlas_323.pt', device)
#model_evaluate.tranformer = nnSpatialTransformer((160,128,112))
model_evaluate.to(device)
model_evaluate.eval()

# set up tensors and permute
input_moving = torch.from_numpy(moving).to(device).float().permute(0, 4, 1, 2, 3)
input_fixed = torch.from_numpy(fixed).to(device).float().permute(0, 4, 1, 2, 3)

# predict
moved, warp = model_evaluate(input_moving, input_fixed, registration=True)

# save moved image

moved = moved.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(moved, 'mice_dataset2/vxm_evaluation_new/323/model_instance_specific_optimized_predicted_atlas_323.nii', fixed_affine)

# save warp
warp = warp.detach().cpu().numpy().squeeze()
vxm.py.utils.save_volfile(warp, 'mice_dataset2/vxm_evaluation_new/323/model_instance_specific_optimized_predicted_atlas_warpfield_323.nii', fixed_affine)
